In [0]:
%sql
CREATE CATALOG IF NOT EXISTS db_test

In [0]:
%sql
USE CATALOG db_test

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS example

In [0]:
%sql
USE SCHEMA example

In [0]:
%sql
CREATE OR REPLACE TABLE silver
(
  device_id INT,
  mrn STRING,
  name STRING,
  time TIMESTAMP,
  heartrate DOUBLE
);

INSERT INTO silver VALUES
  (23, '40580129', 'Nicholas Spears', '2020-02-01T00:01:58.000+0000', 54.0122153343),
  (17, '52804177', 'Lynn Russell',     '2020-02-01T00:02:55.000+0000', 92.5136468131),
  (37, '65300842', 'Samuel Hughes',    '2020-02-01T00:08:58.000+0000', 52.1354807863),
  (23, '40580129', 'Nicholas Spears',  '2020-02-01T00:16:51.000+0000', 54.6477014191),
  (17, '52804177', 'Lynn Russell',     '2020-02-01T00:18:08.000+0000', 95.033344842),
  (37, '65300842', 'Samuel Hughes',    '2020-02-01T00:23:58.000+0000', 57.3391541312),
  (23, '40580129', 'Nicholas Spears',  '2020-02-01T00:31:58.000+0000', 56.6165053697),
  (17, '52804177', 'Lynn Russell',     '2020-02-01T00:32:56.000+0000', 94.8134313932),
  (37, '65300842', 'Samuel Hughes',    '2020-02-01T00:38:54.000+0000', 56.2469995332),
  (23, '40580129', 'Nicholas Spears',  '2020-02-01T00:46:57.000+0000', 54.8372685558);
  


In [0]:
%sql
SELECT * FROM silver

In [0]:
%sql
CREATE OR REPLACE VIEW gold AS (
  SELECT mrn, name, mean(heartrate) as avg_heartrate,date_trunc("DD", time) date 
  FROM silver
  GROUP BY mrn, name, date_trunc("DD", time)
)

In [0]:
%sql
SELECT * FROM gold

In [0]:
%sql
CREATE OR REPLACE FUNCTION mask(x STRING)
  RETURNS STRING
  RETURN concat(repeat("*",length(x)-2),right(x,2))

In [0]:
%sql
DESCRIBE FUNCTION mask

In [0]:
%sql
SELECT mask('sensititve data') AS data

In [0]:
spark.sql("GRANT USE CATALOG ON CATALOG db_test TO analyst_test")

In [0]:
spark.sql("GRANT USE SCHEMA ON SCHEMA example TO analyst_test")

In [0]:
spark.sql("GRANT SELECT ON VIEW gold TO analyst_test")

In [0]:
data = []  # Create a dataframe to store the table values

try:
    # 直接用 Spark SQL 查询表
    result = spark.sql("""
        SELECT *
        FROM db_test.example.gold
    """).collect()

    for row in result:
        # Row -> tuple
        data.append(tuple(row))

    # Create a DataFrame from the list of tuples
    df = spark.createDataFrame(data)

    # Show the DataFrame
    display(df)

except Exception as e:
    print("Error: \n" + str(e))

In [0]:
data = []  # Create a dataframe to store the table values

try:
    # 直接用 Spark SQL 查询表
    result = spark.sql("""
        SELECT *
        FROM db_test.example.silver
    """).collect()

    for row in result:
        # Row -> tuple
        data.append(tuple(row))

    # Create a DataFrame from the list of tuples
    df = spark.createDataFrame(data)

    # Show the DataFrame
    display(df)

except Exception as e:
    print("Error: \n" + str(e))

In [0]:
spark.sql("GRANT EXECUTE ON FUNCTION mask TO analyst_test")

In [0]:
%sql
SHOW GRANTS analyst_test ON FUNCTION mask

In [0]:

try:
    # 直接用 Spark SQL 查询表
    result = spark.sql("""
        SELECT db_test.example.mask('sensitive data') AS data
    """).collect()

    for row in result:
        # Row -> tuple
        data.append(tuple(row))

    # Create a DataFrame from the list of tuples
    df = spark.createDataFrame(data)

    # Show the DataFrame
    display(df)

except Exception as e:
    print("Error: \n" + str(e))

In [0]:
%sql
SELECT * FROM silver

In [0]:
%sql
CREATE FUNCTION mrn_mask(mrn STRING)
  RETURN CASE WHEN is_account_group_member('analyst_test') THEN mrn ELSE 'REDACTED' END;

In [0]:
%sql
ALTER TABLE silver ALTER COLUMN mrn SET MASK mrn_mask;

In [0]:
%sql
SELECT * FROM silver

In [0]:
%sql
ALTER TABLE silver ALTER COLUMN mrn DROP MASK;

In [0]:
%sql 
SELECT * FROM silver;

In [0]:
%sql
SELECT * FROM silver;

In [0]:
%sql
CREATE FUNCTION device_filter(device_id INT)
RETURN IF(is_account_group_member('admin'),true,device_id < 30);

In [0]:
%sql
ALTER TABLE silver SET ROW FILTER device_filter ON (device_id);

In [0]:
%sql
SELECT * FROM silver;

In [0]:
%sql
ALTER TABLE silver DROP ROW FILTER;

In [0]:
%sql
SELECT * FROM silver;

In [0]:
%sql
SHOW TABLES

In [0]:
%sql
SHOW VIEWS

In [0]:
%sql
SHOW SCHEMAS

In [0]:
%sql
SHOW CATALOGS

In [0]:
%sql
SHOW GRANTS ON VIEW gold

In [0]:
%sql
    
SHOW GRANTS ON TABLE silver

In [0]:
%sql 
SHOW GRANTS ON SCHEMA example

In [0]:
%sql
SHOW GRANTS ON CATALOG db_test

In [0]:
%sql
SHOW GRANTS ON FUNCTION mask

In [0]:
spark.sql("REVOKE EXECUTE ON FUNCTION mask FROM `analyst_test`")

In [0]:
%sql
SHOW GRANTS ON FUNCTION mask

In [0]:
spark.sql("REVOKE USE CATALOG on CATALOG `db_test` FROM `analyst_test`")

In [0]:
%sql
SHOW GRANTS ON CATALOG db_test;

In [0]:
%sql
DROP  CATALOG IF EXISTS db_test CASCADE